In [127]:
%load_ext autoreload
%autoreload 2

import os
import pandas as pd
import numpy as np
from pathlib import Path
from pyproj import Transformer
from tqdm import tqdm

from gps_noise_functions import (
    add_stale_locations,
    add_heavy_tail_noise,
    add_swap_latlon,
    add_sign_flip,
    add_rounding,
    add_rounding_variable_decimals,
    add_ring_gaussian_noise,
)

from IPython.display import display

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [102]:
ROOT_PATH = "dataset/scale_2_landscape/"
ROOT_PATH_GPS_NOISY = "dataset/scale_2_landscape/gps_noisy/"

fp_lucas_train_val = os.path.join(ROOT_PATH, "lucas_harmo_cover_exif_nona_fixed_gps_CBN-Med_expanded_essentials_exists_train_val-0.06min.csv")
fp_lucas_train = os.path.join(ROOT_PATH, "lucas_harmo_cover_exif_nona_fixed_gps_CBN-Med_expanded_essentials_exists_train-0.06min.csv")
fp_lucas_val = os.path.join(ROOT_PATH, "lucas_harmo_cover_exif_nona_fixed_gps_CBN-Med_expanded_essentials_exists_val-0.06min.csv")
fp_lucas_test = os.path.join(ROOT_PATH, "glc24_pa_test_private_CBN-med_matching-LUCAS-500m.csv")

df_lucas_train_val = pd.read_csv(fp_lucas_train_val)
df_lucas_test = pd.read_csv(fp_lucas_test)

print(f"LUCAS occurrences Train-Val: {df_lucas_train_val.shape} ({df_lucas_train_val['id'].nunique()} sites)")
display(df_lucas_train_val.head(1))
print(df_lucas_train_val.columns)
print(f"\nLUCAS occurrences Test: {df_lucas_test.shape} ({df_lucas_test['id'].nunique()} sites)")
display(df_lucas_test.head(1))
print(df_lucas_test.columns)

print("\nExcluding Test surveyIds from Train-val...")
df_lucas_train_val = df_lucas_train_val[~(df_lucas_train_val['id'].isin(df_lucas_test['id'].unique()))]
print(f"LUCAS occurrences Train-Val (after purge of Test): {df_lucas_test.shape} ({df_lucas_test['id'].nunique()} sites)")

relevant_cols_test = ['PlotObservationID_eva', 'surveyId', 'lon', 'lat', 'lucas_matching_ids']

LUCAS occurrences Train-Val: (47258, 12) (7903 sites)


,Unnamed: 0,id,file_path,full_path,image_source,exists,full_path_missing,full_path_2022,full_path_cover,lon,lat,subset
0,0,967805,2009/FR/378/823/37882398N.jpg,LUCAS/2009/FR/378/823/37882398N.jpg,file_path_gisco_north,False,LUCAS/../../lucas_missing/2009/FR/378/823/3788...,LUCAS/../../lucas_photos_all_2022/2009/FR/378/...,LUCAS/../../lucas_cover/2009/FR/378/823/378823...,3.303906,44.47791,train


Index(['Unnamed: 0', 'id', 'file_path', 'full_path', 'image_source', 'exists',
       'full_path_missing', 'full_path_2022', 'full_path_cover', 'lon', 'lat',
       'subset'],
      dtype='object')

LUCAS occurrences Test: (1252, 27) (64 sites)


,PlotObservationID_eva,lon,lat,year,datasetName,Access.regime,Expert.System,Cover.abundance.scale,geoUncertaintyInM,observer,...,region,surveyID,country,speciesId,surveyId,split,id,x_EPSG_32631,y_EPSG_32631,lucas_matching_ids
0,1953693.0,6.215064,43.12536,2021,CBNMed,2,V32,Braun/Blanquet (old),3.0,NaN,...,MEDITERRANEAN,93212,France,10822.0,74414,train,74414,761530.363056,4.779755e+06,751990


Index(['PlotObservationID_eva', 'lon', 'lat', 'year', 'datasetName',
       'Access.regime', 'Expert.System', 'Cover.abundance.scale',
       'geoUncertaintyInM', 'observer', 'areaInM2', 'coverTreeLayer',
       'coverShrubLayer', 'coverHerbLayer', 'coverMossLayer', 'source',
       'access', 'region', 'surveyID', 'country', 'speciesId', 'surveyId',
       'split', 'id', 'x_EPSG_32631', 'y_EPSG_32631', 'lucas_matching_ids'],
      dtype='object')

Excluding Test surveyIds from Train-val...
LUCAS occurrences Train-Val (after purge of Test): (1252, 27) (64 sites)


In [121]:
def delta_after_shift(d, lon, lat):
    """Computes the delta between 2 WGS84 coords based on a metric distance, through the EPSG:3035.
    
    WARNING: extremely slow to call within a loop.
    
    d: distance in meters (added to both x and y in EPSG:3035)
    lon, lat: original WGS84 coordinates (degrees)

    Returns (delta_lon, delta_lat) between before and after.
    """
    wgs84 = "EPSG:4326"    # lon/lat in WGS84
    epsg3035 = "EPSG:3035" # ETRS89 / LAEA Europe

    # always_xy=True → inputs/outputs are (lon, lat) for geographic CRS
    to_3035 = Transformer.from_crs(wgs84, epsg3035, always_xy=True)
    to_4326 = Transformer.from_crs(epsg3035, wgs84, always_xy=True)

    # 1) WGS84 → EPSG:3035
    x0, y0 = to_3035.transform(lon, lat)

    # 2) Add distance d to each projected coordinate
    x1 = x0 + d
    y1 = y0 + d

    # 3) Back to WGS84
    lon1, lat1 = to_4326.transform(x1, y1)

    # 4) Delta between before and after
    dlon = lon1 - lon
    dlat = lat1 - lat

    return dlon, dlat

# Old, deprecated
def apply_noise_over_GPS_in_meters(
    df: pd.DataFrame,
    gamma: float = 0.1,
    epsilon: int = 100,
    random_state: int = None
):
    """
    Apply random noise to a pandas column.

    Args:
        df (pd.DataFrame): Input dataframe.
        column (str): Column name to modify.
        gamma (float): Probability of applying noise to each value (0 to 1).
        epsilon (float): Noise magnitude.
        noise_type (str): Type of noise to apply ("gaussian" or "uniform").
        random_state (int, optional): Seed for reproducibility.

    Returns:
        pd.DataFrame: New dataframe with the noisy column.
    """
    if random_state is not None:
        np.random.seed(random_state)

    df_noisy = df.copy()
    
    wgs84 = "EPSG:4326"    # lon/lat in WGS84
    epsg3035 = "EPSG:3035" # ETRS89 / LAEA Europe
    # always_xy=True → inputs/outputs are (lon, lat) for geographic CRS
    to_3035 = Transformer.from_crs(wgs84, epsg3035, always_xy=True)
    to_4326 = Transformer.from_crs(epsg3035, wgs84, always_xy=True)

    # Decide which rows get noise
    mask = np.random.rand(len(df)) < gamma
    signs_lon = np.random.choice([-1, 1], size=len(df))
    signs_lat = np.random.choice([-1, 1], size=len(df))

    # Apply noise only where mask is True
    df_noisy['lon_noisy'] = df['lon'].copy()
    df_noisy['lat_noisy'] = df['lat'].copy()
    df_noisy['gps_match'] = [True]*len(df)
    for (rowi, row), maski in tqdm(zip(df.iterrows(), mask), total=(len(df))):
        if maski:
            lon = row['lon']
            lat = row['lat']
            x0, y0 = to_3035.transform(lon, lat)
            # 2) Add distance d to each projected coordinate
            x1 = x0 + epsilon
            y1 = y0 + epsilon
            # 3) Back to WGS84
            lon1, lat1 = to_4326.transform(x1, y1)
            # 4) Delta between before and after
            d_lon = lon1 - lon
            d_lat = lat1 - lat

            df_noisy.loc[rowi, 'lon_noisy'] += signs_lon[rowi] * d_lon
            df_noisy.loc[rowi, 'lat_noisy'] += signs_lat[rowi] * d_lat
            df_noisy.loc[rowi, 'gps_match'] = False
       
    return df_noisy

gps_error_probability = 0.25
gps_error_in_meters = 1000

In [132]:
def init_noisy(df):
    df = df.copy()
    df["lon_noisy"] = df["lon"].values
    df["lat_noisy"] = df["lat"].values
    return df

def simulate_gps_noise_mixture(
    df,
    gamma=0.10,
    mechanism_probs=None,
    seed=42,
    tol=1e-9
):
    """
    Simulate GPS noise with a mixture model:
    - gamma fraction of points are noised
    - each selected point gets exactly ONE mechanism drawn from mechanism_probs
    """

    if mechanism_probs is None:
        mechanism_probs = {
            "stale": 0.2,
            # "heavy_tail": 0.2,  # Risk of exploding with inf values because of lognormal
            "gaussian_beyond_ring": 0.2,
            "swap": 0.2,
            "sign_flip": 0.2,
            "rounding": 0.2,
        }

    assert abs(sum(mechanism_probs.values()) - 1.0) < 1e-6

    rng = np.random.default_rng(seed)
    df = init_noisy(df)

    n = len(df)

    # Select samples to add noise to
    mask_noisy = rng.random(n) < gamma
    noisy_idx = np.where(mask_noisy)[0]

    df["noise_type"] = "none"
    if len(noisy_idx) == 0:
        df["gps_match"] = 0
        df["gps_noised"] = 0
        return df

    # Assign a noise for each sample
    mechanisms = list(mechanism_probs.keys())
    probs = list(mechanism_probs.values())

    assigned = rng.choice(mechanisms, size=len(noisy_idx), p=probs)  # Get a list of mecanisms based on their probability distribution defined in mecanism_probs

    # Apply mecanisms
    df_out = df.copy()

    for mech in mechanisms:
        idx = noisy_idx[assigned == mech]
        if len(idx) == 0:
            continue

        sub_df = df_out.iloc[idx].copy()

        if mech == "stale":
            sub_df = add_stale_locations(sub_df, p=1.0, seed=seed)
        elif mech == "heavy_tail":
            sub_df = add_heavy_tail_noise(sub_df, p=1.0, median_m=100000, seed=seed+1)  # median_m is the radius in meters in which the coordinates can be degraded, following a log-normal curve
        elif mech == "swap":
            sub_df = add_swap_latlon(sub_df, p=1.0, seed=seed+2)  # pseudo-random
        elif mech == "sign_flip":
            sub_df = add_sign_flip(sub_df, p=1.0, seed=seed+3)  # pseudo-random
        elif mech == "rounding":
            sub_df = add_rounding_variable_decimals(sub_df, p=1.0, seed=seed+4)  #
        elif mech == "gaussian_beyond_ring":
            sub_df = add_ring_gaussian_noise(sub_df, noise_meters=100000, seed=None, noise_version='beyond_ring')
        elif mech == "gaussian_inside_ring":
            sub_df = add_ring_gaussian_noise(sub_df, noise_meters=100000, seed=None, noise_version='inside_ring')

        # réinjecter les résultats
        df_out.loc[idx, "lon_noisy"] = sub_df["lon_noisy"].values
        df_out.loc[idx, "lat_noisy"] = sub_df["lat_noisy"].values
        
        # Record which mechanism was used
        df_out.loc[idx, "noise_type"] = mech

    # Add label columns
    lon_match = np.isclose(df_out["lon"], df_out["lon_noisy"], atol=tol)
    lat_match = np.isclose(df_out["lat"], df_out["lat_noisy"], atol=tol)

    df_out["gps_match"] = (lon_match & lat_match).astype(int)
    df_out["gps_noised"] = 1 - df_out["gps_match"]

    return df_out

seeds = [1, 2, 3, 5, 8, 13, 21, 34, 55, 89]  # Fibonacci sequence

In [123]:
fp_lucas_train_val_noisy = os.path.join(ROOT_PATH_GPS_NOISY, Path(fp_lucas_train_val).name)
fp_lucas_train_noisy = os.path.join(ROOT_PATH_GPS_NOISY, Path(fp_lucas_train).name)
fp_lucas_val_noisy = os.path.join(ROOT_PATH_GPS_NOISY, Path(fp_lucas_val).name)
fp_lucas_test_noisy = os.path.join(ROOT_PATH_GPS_NOISY, Path(fp_lucas_test).name)

df_lucas_test_unique_sId = df_lucas_test[relevant_cols_test].drop_duplicates(subset='surveyId', keep="first", ignore_index=True)
df_lucas_test_unique_sId['id'] = df_lucas_test_unique_sId['surveyId']

for seed in tqdm(seeds, 'Random seeds processed'):
    # Apply noise
    df_lucas_train_val_noisy_gps = simulate_gps_noise_mixture(df_lucas_train_val, gamma=0.50, seed=seed)
    df_lucas_test_noisy_gps = simulate_gps_noise_mixture(df_lucas_test_unique_sId, gamma=0.50, seed=seed)
    
    # Save files
    df_lucas_train_val_noisy_gps.to_csv(f"{fp_lucas_train_val_noisy.split('.csv')[0]}_noise_mixture_seed{seed}.csv", index=False)
    df_lucas_train_val_noisy_gps[df_lucas_train_val_noisy_gps['subset'] == 'train'].to_csv(f"{fp_lucas_train_noisy.split('.csv')[0]}_noise_mixture_seed{seed}.csv", index=False)
    df_lucas_train_val_noisy_gps[df_lucas_train_val_noisy_gps['subset'] == 'val'].to_csv(f"{fp_lucas_val_noisy.split('.csv')[0]}_noise_mixture_seed{seed}.csv", index=False)
    df_lucas_test_noisy_gps.to_csv(f"{fp_lucas_test_noisy.split('.csv')[0]}_noise_mixture_seed{seed}.csv", index=False)

print(f'Train-val size: {df_lucas_train_val_noisy_gps.shape}')
display(df_lucas_train_val_noisy_gps.sample(3)[['id', 'lon', 'lon_noisy', 'noise_type', 'gps_match', 'gps_noised']])
print(f'Test size: {df_lucas_test_noisy_gps.shape}')
display(df_lucas_test_noisy_gps.sample(3)[['id', 'lon', 'lon_noisy', 'noise_type', 'gps_match', 'gps_noised']])

Random seeds processed: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 10/10 [00:07<00:00,  1.33it/s]

Train-val size: (47258, 17)


,id,lon,lon_noisy,noise_type,gps_match,gps_noised
43924,403666,2.514534,2.514534,none,1,0
37586,1069649,4.561840,4.561840,none,1,0
24170,734398,4.049490,4.049490,none,1,0


Test size: (64, 11)


,id,lon,lon_noisy,noise_type,gps_match,gps_noised
0,74414,6.215064,6.215064,none,1,0
33,1670585,3.222458,43.442031,swap,0,1
24,1163395,6.182538,7.050960,stale,0,1


## Insert file paths in test set

In [116]:
fp_lucas_train_val_noisy = os.path.join(ROOT_PATH_GPS_NOISY, 'lucas_harmo_cover_exif_nona_fixed_gps_CBN-Med_expanded_essentials_exists_train_val-0.06min_noise_mixture_seed1.csv')
fp_lucas_test_noisy =  os.path.join(ROOT_PATH_GPS_NOISY, 'glc24_pa_test_private_CBN-med_matching-LUCAS-500m_noise_mixture_seed1.csv')

df_train_val_noisy = pd.read_csv(fp_lucas_train_val_noisy)
df_test_noisy = pd.read_csv(fp_lucas_test_noisy)

print(df_train_val_noisy.columns)
print(df_test_noisy.columns)
print(df_test_noisy.sample(1)['lucas_matching_ids'])

Index(['Unnamed: 0', 'id', 'file_path', 'full_path', 'image_source', 'exists',
       'full_path_missing', 'full_path_2022', 'full_path_cover', 'lon', 'lat',
       'subset', 'lon_noisy', 'lat_noisy', 'noise_type', 'gps_match',
       'gps_noised'],
      dtype='object')
Index(['PlotObservationID_eva', 'surveyId', 'lon', 'lat', 'lucas_matching_ids',
       'id', 'lon_noisy', 'lat_noisy', 'noise_type', 'gps_match',
       'gps_noised'],
      dtype='object')
3    1021041 1016806 1011135 
Name: lucas_matching_ids, dtype: object


In [120]:
for seed in tqdm(seeds, 'Random seeds processed'):
    fp_lucas_test_noisy =  os.path.join(ROOT_PATH_GPS_NOISY, f'glc24_pa_test_private_CBN-med_matching-LUCAS-500m_noise_mixture_seed{seed}.csv')
    df_test_noisy = pd.read_csv(fp_lucas_test_noisy)
    
    df_test_noisy['file_path'] = ['']*len(df_test_noisy)
    for rowi, row in tqdm(df_test_noisy.copy().iterrows(), total=df_test_noisy.copy().shape[0]):
        fps_lucas = ''
        lucas_ids = row['lucas_matching_ids'].split()
        for k, lucas_id in enumerate(lucas_ids):
            df_slice = df_train_val_noisy[df_train_val_noisy['id'] == int(lucas_id)]
            file_paths = df_slice['file_path'].tolist()
            if file_paths == []:
                print(f"[WARNING] Lucas ID {df_slice['id']} not found")
            fps_lucas += ' '.join(file_paths)
            fps_lucas += ';' if len(lucas_ids) > 1 and k < len(lucas_ids)-1 else ''
        df_test_noisy.loc[rowi, 'lucas_matching_ids'] = ';'.join(lucas_ids)
        df_test_noisy.loc[rowi, 'file_path'] = fps_lucas
    
    df_test_noisy_surveyIds_groupby = df_test_noisy.drop_duplicates(subset="surveyId", keep="first")
    df_test_noisy.to_csv(fp_lucas_test_noisy, index=False)

display(df_test_noisy.sample(3)[['lon', 'lat', 'id', 'lucas_matching_ids', 'file_path']])

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 64/64 [00:00<00:00, 2027.66it/s]

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 64/64 [00:00<00:00, 1983.64it/s]

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 64/64 [00:00<00:00, 1935.38it/s]

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████

,lon,lat,id,lucas_matching_ids,file_path
56,6.210201,43.125819,3250325,751990,2012/FR/401/222/40122232N.jpg 2012/FR/401/222/...
43,3.229558,43.427340,2265612,914102;913730;907186;519111,2009/FR/377/222/37722282N.jpg 2009/FR/377/222/...
50,5.151430,43.751700,2735525,1095053;1086393;1081666,2009/FR/393/023/39302306N.jpg 2009/FR/393/023/...
